# training-step-cycle — ex2: diagnose a buggy training step that forgets zero_grad

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `training-step-cycle`. Running the final beacon cell reports progress against the `PyTorch: Training step cycle` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Training step cycle` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`training-step-cycle`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "training-step-cycle"
DD_SUBTOPIC = "PyTorch: Training step cycle"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## The PyTorch training step cycle — quick refresher

Every PyTorch training loop body, no matter how exotic the model, walks the same four-call cycle on each batch:

```
logits = model(x)             # 1. forward
loss   = loss_fn(logits, y)   # 2. loss
loss.backward()               # 3. backward → gradients into .grad
optimizer.step()              # 4. apply update from .grad to params
optimizer.zero_grad()         # 5. clear .grad so next batch starts fresh
```

**Order matters.** `.backward()` must come before `.step()` (no grads → no update). `.zero_grad()` must come after `.step()` (or before the next forward) — otherwise gradients from the previous batch accumulate into the next one. The default is gradient ACCUMULATION; `.zero_grad()` is what makes each batch independent.

### Exercise 2 — diagnose a buggy training step that forgets zero_grad

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze a training loop that omits `optimizer.zero_grad()` and fix it by inserting the missing call in the correct position so the loss curve becomes monotonically decreasing.
> Keywords: debug, gradient-accumulation, zero-grad-missing
> ```

**KCs targeted:** `training-step-zero-grad-resets-accumulation`, `training-step-debug-via-loss-trajectory`

Below is `train_buggy` — a training loop whose loss curve oscillates wildly and never gets close to zero. The bug is that it is missing `optimizer.zero_grad()`. Implement `ex2_train_fixed(w_init, x, y, lr, n_steps)` — the corrected version. Use exactly the same 5-call cycle as exercise 1, but this time return a tuple `(w_final, losses, buggy_losses)` where `buggy_losses` is what the buggy version produces on the same inputs.

You must define `ex2_train_fixed` AND call the provided `train_buggy` to populate `buggy_losses`. The test then verifies your fixed version converges while the buggy one does not.

Use the same problem as ex1 (`y = 2x`, lr=0.05).

```python
def train_buggy(w_init, x, y, lr, n_steps):
    # BUG: never calls optimizer.zero_grad()
    w = t.tensor([w_init], requires_grad=True)
    optimizer = t.optim.SGD([w], lr=lr)
    losses = []
    for _ in range(n_steps):
        pred = w * x
        loss = ((pred - y) ** 2).mean()
        losses.append(loss.item())
        loss.backward()
        optimizer.step()
        # <-- missing optimizer.zero_grad() here
    return w.detach().clone(), losses
```

In [ ]:
def train_buggy(w_init, x, y, lr, n_steps):
    # BUG: never calls optimizer.zero_grad()
    w = t.tensor([w_init], requires_grad=True)
    optimizer = t.optim.SGD([w], lr=lr)
    losses = []
    for _ in range(n_steps):
        pred = w * x
        loss = ((pred - y) ** 2).mean()
        losses.append(loss.item())
        loss.backward()
        optimizer.step()
    return w.detach().clone(), losses


def ex2_train_fixed(w_init, x, y, lr, n_steps):
    # Run the buggy version first to capture its loss trajectory.
    _, buggy_losses = train_buggy(w_init, x, y, lr, n_steps)

    # Fixed version — adds the missing zero_grad after step.
    w = t.tensor([w_init], requires_grad=True)
    optimizer = t.optim.SGD([w], lr=lr)
    losses = []
    for _ in range(n_steps):
        pred = w * x
        loss = ((pred - y) ** 2).mean()
        losses.append(loss.item())
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()      # <-- the fix
    return w.detach().clone(), losses, buggy_losses


<details><summary>Solution</summary>

```python
def train_buggy(w_init, x, y, lr, n_steps):
    # BUG: never calls optimizer.zero_grad()
    w = t.tensor([w_init], requires_grad=True)
    optimizer = t.optim.SGD([w], lr=lr)
    losses = []
    for _ in range(n_steps):
        pred = w * x
        loss = ((pred - y) ** 2).mean()
        losses.append(loss.item())
        loss.backward()
        optimizer.step()
    return w.detach().clone(), losses


def ex2_train_fixed(w_init, x, y, lr, n_steps):
    # Run the buggy version first to capture its loss trajectory.
    _, buggy_losses = train_buggy(w_init, x, y, lr, n_steps)

    # Fixed version — adds the missing zero_grad after step.
    w = t.tensor([w_init], requires_grad=True)
    optimizer = t.optim.SGD([w], lr=lr)
    losses = []
    for _ in range(n_steps):
        pred = w * x
        loss = ((pred - y) ** 2).mean()
        losses.append(loss.item())
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()      # <-- the fix
    return w.detach().clone(), losses, buggy_losses
```

**Why the buggy version OSCILLATES instead of just stalling.** Without `zero_grad`, the gradient at step N is the SUM of every per-batch gradient computed so far. After the first step the (now oversized) accumulated gradient overshoots the minimum. At the new point on the loss surface the gradient points the OTHER way; that new gradient is added to the previous accumulated value, partially cancelling it. The result is a noisy ping-pong around the minimum — the loss neither converges to zero nor blows up to infinity, it just wanders. On harder problems with steeper curvature the same bug can outright diverge.

**Diagnosis recipe.** When your training loss INCREASES at any step early in training, the first three things to check are (in order): (1) is `zero_grad` missing? (2) is the learning rate too high? (3) is the loss being aggregated correctly (e.g. summed instead of meaned across the batch)? The `zero_grad` bug has a distinctive signature: many non-monotone loss steps and a loss floor that never gets below the starting value.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()